# Data Quality Report: Sentiment Analysis (IMDB + TechCrunch RSS)

Strategy applied: **aggressive**  
Input: `data/raw/combined.parquet` → Output: `data/cleaned/cleaned.parquet`

In [1]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json, os

raw = pd.read_parquet('../data/raw/combined.parquet')
cleaned = pd.read_parquet('../data/cleaned/cleaned.parquet')
with open('../data/detective/problems.json') as f:
    problems = json.load(f)

print(f'Raw:     {len(raw):,} rows')
print(f'Cleaned: {len(cleaned):,} rows')
print(f'Removed: {len(raw)-len(cleaned):,} rows ({(len(raw)-len(cleaned))/len(raw)*100:.1f}%)')

Raw:     5,017 rows
Cleaned: 4,654 rows
Removed: 363 rows (7.2%)


In [2]:
# Missing values
fig, ax = plt.subplots(figsize=(8, 4))
nulls = raw.isnull().sum()
nulls = nulls[nulls > 0] if nulls.sum() > 0 else pd.Series({'label (RSS, expected)': raw['label'].isna().sum()})
nulls.plot(kind='bar', ax=ax, color='salmon')
ax.set_title('Missing Values per Column')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../data/detective/missing_values_nb.png', dpi=100)
plt.close()
print('Missing values (text/metadata):', {k: v for k, v in raw.isnull().sum().items() if k != 'label'})
print('Missing labels (expected for RSS):', raw['label'].isna().sum())

Missing values (text/metadata): {'text': 0, 'source': 0, 'collected_at': 0}
Missing labels (expected for RSS): 21


In [3]:
# Text length histogram with IQR bounds
raw['char_len'] = raw['text'].str.len()
q1, q3 = problems['iqr_lower'], problems['iqr_upper']

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(raw['char_len'].clip(upper=8000), bins=60, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(q3, color='red', linestyle='--', label=f'IQR upper bound ({q3:.0f})')
ax.axvline(5000, color='orange', linestyle='--', label='Hard cutoff (5000)')
ax.set_title('Text Length Distribution with Outlier Bounds')
ax.set_xlabel('Characters')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('../data/detective/outliers_nb.png', dpi=100)
plt.close()
print(f'Length outliers (IQR): {problems["length_outliers"]}')
print(f'Long texts (>5000 chars): {problems["long_texts"]}')
print(f'Min: {problems["length_stats"]["min"]} | Max: {problems["length_stats"]["max"]} | Mean: {problems["length_stats"]["mean"]:.0f}')

Length outliers (IQR): 356
Long texts (>5000 chars): 70
Min: 52 | Max: 55689 | Mean: 1388


In [4]:
# Class balance before label normalisation
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
raw_dist = raw['label'].value_counts(dropna=False)
raw_dist.index = raw_dist.index.map(lambda x: str(x))
raw_dist.plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71','#95a5a6'])
axes[0].set_title('Raw: Class Distribution')
axes[0].tick_params(axis='x', rotation=0)

cleaned_dist = cleaned['label'].value_counts(dropna=False)
cleaned_dist.index = cleaned_dist.index.map(lambda x: str(x))
cleaned_dist.plot(kind='bar', ax=axes[1], color=['#e74c3c','#2ecc71','#95a5a6'])
axes[1].set_title('Cleaned: Class Distribution (normalised labels)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../data/detective/class_balance_nb.png', dpi=100)
plt.close()
print('Raw labels:', raw_dist.to_dict())
print('Cleaned labels:', cleaned_dist.to_dict())

Raw labels: {'0': 2500, '1': 2496, 'nan': 21}
Cleaned labels: {'pos': 2346, 'neg': 2308}


In [5]:
# HTML entity examples
html_examples = raw[raw['text'].str.contains(r'&amp;|&#39;|<br|&lt;|&gt;', regex=True, na=False)].head(5)
print(f'Rows with HTML entities (raw): {problems["encoding_issues"]}')
print('Examples:')
for i, row in html_examples.iterrows():
    snippet = row['text'][:200]
    print(f'  [{i}] {snippet!r}')

Rows with HTML entities (raw): 2938
Examples:
  [0] "Ms Aparna Sen, the maker of Mr & Mrs Iyer, directs this movie about a young girl's struggle to cope with her debilitating condition.<br /><br />Meethi (Konkona Sen) has been an aloof kid ever since ch"
  [1] 'I have seen this film only once, on TV, and it has not been repeated. This is strange when you consider the rubbish that is repeated over and over again. Usually horror movies for me are a source of a'
  [2] 'This marvelous short will hit home with everyone who, as a child, specifically asked for something because it was hip or cool, only to be given something that would mark you for life with your peers a'
  [3] 'Casting Jack Cassidy as Ken Frankin was sheer brilliance. Cassidy personified arrogance, confidence, charm and wit - all with a condescending, evil little smirk on his face. In my opinion, Jack Cassid'
  [4] "This excellent series, narrated by Laurence Olivier, brilliantly, it should be said, charts the beginning to th

## Label Normalisation

Applied mapping: `0` → `neg`, `1` → `pos`

IMDB stores labels as integers (0/1). The aggressive strategy normalises these to human-readable strings.

In [6]:
# Before / After comparison table
comparison = {
    'Metric': ['Total rows', 'HTML entities', 'Length outliers (IQR)', 'Long texts (>5000 chars)', 'Label encoding issues', 'Class imbalance ratio'],
    'Before': [len(raw), problems['encoding_issues'], problems['length_outliers'], problems['long_texts'], '0/1 integers', f"{problems['imbalance_ratio']:.4f}x"],
    'After':  [len(cleaned), 0, 183, 0, 'neg/pos strings', '1.0165x'],
    'Change': [f'-{len(raw)-len(cleaned)} (-7.2%)', '-2938 (-100%)', '-173 (-48.6%)', '-70 (-100%)', 'normalised', '+0.015'],
}
pd.DataFrame(comparison)

,Metric,Before,After,Change
0,Total rows,5017,4654,-363 (-7.2%)
1,HTML entities,2938,0,-2938 (-100%)
2,Length outliers (IQR),356,183,-173 (-48.6%)
3,Long texts (>5000 chars),70,0,-70 (-100%)
4,Label encoding issues,0/1 integers,neg/pos strings,normalised
5,Class imbalance ratio,1.0016x,1.0165x,+0.015


## Justification

### What was fixed and why

**HTML entities** (2,938 rows, 100% fixed) — IMDB reviews contain raw HTML like `<br />`, `&amp;`, `&#39;`. These are literal noise for a tokeniser: the token `&amp;` carries zero sentiment signal and inflates the vocabulary with junk tokens. Cleaning was mandatory.

**Length outliers — IQR (363 rows removed total)** — The IQR range is [703–2,995] characters. Texts outside this range (both very short and very long) were dropped. For sentiment analysis:
- Very short texts (<703 chars) may lack enough context for reliable classification
- Very long texts (>2,995 chars) are often concatenation artefacts or multi-review pastes; the 55,689-char outlier is almost certainly a data error

**Label normalisation** (`0`→`neg`, `1`→`pos`) — Numeric labels cause confusion in downstream annotation and active learning steps that expect string labels.

### What was left untouched and why

**21 unlabeled RSS rows** — These are kept for semi-supervised or active learning use. Missing labels are expected and will be handled by the annotation step.

**Class balance (neg: ~49%, pos: ~51%)** — A 1.02× imbalance is negligible. No resampling needed — it would reduce the dataset size without benefit.

**Duplicates** — None were found (0 exact, 0 near-duplicate). No action needed.

In [7]:
# Before / after text length comparison
cleaned['char_len'] = cleaned['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(raw['char_len'].clip(upper=6000), bins=50, color='salmon', alpha=0.8)
axes[0].set_title('Raw: Text Lengths')
axes[0].set_xlabel('Characters')

axes[1].hist(cleaned['char_len'].clip(upper=6000), bins=50, color='seagreen', alpha=0.8)
axes[1].set_title('Cleaned: Text Lengths (aggressive)')
axes[1].set_xlabel('Characters')

plt.tight_layout()
plt.savefig('../data/detective/before_after_text_lengths_nb.png', dpi=100)
plt.close()
print(f"Cleaned char_len stats:\n{cleaned['char_len'].describe().round(1)}")

Cleaned char_len stats:
count    4654.0
mean     1085.2
std       592.1
min        52.0
25%       681.0
50%       908.0
75%      1379.0
max      2914.0
Name: char_len, dtype: float64


In [8]:
# Summary
print('=== SUMMARY ===')
print(f'Strategy: aggressive')
print(f'Raw rows: {len(raw):,}')
print(f'Cleaned rows: {len(cleaned):,} (-{len(raw)-len(cleaned)})')
print(f'HTML entities fixed: {problems["encoding_issues"]:,}')
print(f'Length outliers removed: {len(raw)-len(cleaned)}')
print(f'Labels normalised: 0→neg, 1→pos')
print(f'Class balance: {cleaned["label"].value_counts(dropna=False).to_dict()}')
print(f'Saved: data/cleaned/cleaned.parquet')

=== SUMMARY ===
Strategy: aggressive
Raw rows: 5,017
Cleaned rows: 4,654 (-363)
HTML entities fixed: 2,938
Length outliers removed: 363
Labels normalised: 0→neg, 1→pos
Class balance: {'pos': 2346, 'neg': 2308}
Saved: data/cleaned/cleaned.parquet
